# 01 · Build an autograd engine (micrograd-style)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/01_micrograd.ipynb) &nbsp; 📖 Concept deck: [Backpropagation](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/03-backpropagation/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a scalar `Value` class that records the computational graph, local backward rules for +, ×, tanh and powers, and a topological-order backward pass that matches finite differences.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import math

class Value:
    """A scalar that remembers how it was computed."""
    def __init__(self, data, parents=(), op=''):
        self.data, self.grad = float(data), 0.0
        self._parents, self._op = parents, op
        self._backward = lambda: None
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## Task 1: Addition and multiplication with backward rules

Write `add(a, b)` and `mul(a, b)` that return a new `Value` whose `_backward` adds the right local gradient into its parents:
for $c = a + b$: $\partial c/\partial a = \partial c/\partial b = 1$; for $c = a \cdot b$: $\partial c/\partial a = b$, $\partial c/\partial b = a$.
Always use `+=`: a value used twice must collect gradient from both uses.

In [ ]:
def add(a, b):
    out = Value(a.data + b.data, (a, b), '+')
    def _backward():
        # TODO: a.grad += ... ; b.grad += ...
        raise NotImplementedError
    out._backward = _backward
    return out

def mul(a, b):
    # TODO
    raise NotImplementedError

def add_mul(a, b):
    return add, mul

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_add_mul(f):
    add_, mul_ = f(None, None)
    a, b = Value(2.0), Value(-3.0)
    c = mul_(a, b); c.grad = 1.0; c._backward()
    assert c.data == -6 and a.grad == -3 and b.grad == 2, "d(ab)/da = b, d(ab)/db = a"
    x = Value(5.0); y = add_(x, x); y.grad = 1.0; y._backward()
    assert x.grad == 2, "x used twice must receive gradient twice (use +=)"

check("add_mul", _test_add_mul, add_mul)

## Task 2: tanh and powers

Add `tanh(a)` with $\frac{d}{da}\tanh a = 1 - \tanh^2 a$, and `pow_(a, k)` for a constant exponent k with $\frac{d}{da} a^k = k a^{k-1}$.

In [ ]:
def tanh(a):
    # TODO
    raise NotImplementedError

def pow_(a, k):
    # TODO
    raise NotImplementedError

def tanh_pow(a=None):
    return tanh, pow_

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_tanh_pow(f):
    tanh_, pow__ = f()
    a = Value(0.5); t = tanh_(a); t.grad = 1.0; t._backward()
    assert np.isclose(t.data, np.tanh(0.5)) and np.isclose(a.grad, 1 - np.tanh(0.5) ** 2)
    b = Value(3.0); p = pow__(b, 2); p.grad = 1.0; p._backward()
    assert p.data == 9 and b.grad == 6

check("tanh_pow", _test_tanh_pow, tanh_pow)

## Task 3: Backward in topological order

`backward(root)` must (1) build a topological order of all nodes reachable from `root` (each node after all its parents), (2) set `root.grad = 1`, and (3) call `_backward()` on nodes in **reverse** topological order, so every node's gradient is complete before it is pushed to its parents.

In [ ]:
def backward(root):
    order, seen = [], set()
    # TODO: depth-first search over _parents to fill `order`, then reverse pass
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_backward(f):
    def _op(data, parents, rule):
        out = Value(data, parents)
        def bw():
            for p, g in zip(parents, rule(out)): p.grad += g * out.grad
        out._backward = bw; return out
    A = lambda a, b: _op(a.data + b.data, (a, b), lambda o: (1, 1))
    M = lambda a, b: _op(a.data * b.data, (a, b), lambda o: (b.data, a.data))
    T = lambda a: _op(math.tanh(a.data), (a,), lambda o: (1 - o.data ** 2,))
    P = lambda a, k: _op(a.data ** k, (a,), lambda o: (k * a.data ** (k - 1),))
    # a neuron: tanh(w1*x1 + w2*x2 + b), and a squared error
    x1, x2, w1, w2, b = Value(2.0), Value(0.0), Value(-3.0), Value(1.0), Value(6.8813735870195432)
    out = T(A(A(M(x1, w1), M(x2, w2)), b)); loss = P(A(out, Value(-1.0)), 2)
    f(loss)
    def L(w1v):
        return (math.tanh(2 * w1v + 0 + 6.8813735870195432) - 1) ** 2
    num = (L(-3 + 1e-6) - L(-3 - 1e-6)) / 2e-6
    assert abs(w1.grad - num) < 1e-5, f"dloss/dw1 = {w1.grad}, finite difference {num}"
    y = Value(3.0); z = M(y, y); f(z)
    assert y.grad == 6, "y*y: both uses contribute"

check("backward", _test_backward, backward)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Addition and multiplication with backward rules
def add_solution_(a, b):
    out = Value(a.data + b.data, (a, b), '+')
    def _backward():
        a.grad += out.grad; b.grad += out.grad
    out._backward = _backward
    return out

def mul_solution_(a, b):
    out = Value(a.data * b.data, (a, b), '*')
    def _backward():
        a.grad += b.data * out.grad; b.grad += a.data * out.grad
    out._backward = _backward
    return out

def add_mul_solution(a, b):
    return add_solution_, mul_solution_

check("add_mul (reference)", _test_add_mul, add_mul_solution)

In [ ]:
# @title Solution for Task 2: tanh and powers
def tanh_solution_(a):
    t = math.tanh(a.data); out = Value(t, (a,), 'tanh')
    def _backward():
        a.grad += (1 - t * t) * out.grad
    out._backward = _backward
    return out

def pow_solution_(a, k):
    out = Value(a.data ** k, (a,), f'**{k}')
    def _backward():
        a.grad += k * a.data ** (k - 1) * out.grad
    out._backward = _backward
    return out

def tanh_pow_solution(a=None):
    return tanh_solution_, pow_solution_

check("tanh_pow (reference)", _test_tanh_pow, tanh_pow_solution)

In [ ]:
# @title Solution for Task 3: Backward in topological order
def backward_solution(root):
    order, seen = [], set()
    def visit(v):
        if id(v) in seen: return
        seen.add(id(v))
        for p in v._parents: visit(p)
        order.append(v)
    visit(root)
    root.grad = 1.0
    for v in reversed(order): v._backward()

check("backward (reference)", _test_backward, backward_solution)

---
## Experiments
**Train one neuron.** Use your engine to fit tanh(w·x + b) to a few points by gradient descent: zero the grads, forward, backward, step.

In [ ]:
A, M = add_mul_solution(None, None); T, P = tanh_pow_solution()
xs, ys = [-2.0, -1.0, 1.0, 2.0], [-0.9, -0.6, 0.6, 0.9]
w, b = Value(0.1), Value(0.0)
for step in range(201):
    loss = Value(0.0)
    for x, y in zip(xs, ys):
        loss = A(loss, P(A(T(A(M(w, Value(x)), b)), Value(-y)), 2))
    w.grad = b.grad = 0.0
    backward_solution(loss)
    w.data -= 0.1 * w.grad; b.data -= 0.1 * b.grad
    if step % 50 == 0: print(f"step {step:3d}: loss {loss.data:.4f}  w={w.data:.3f} b={b.data:.3f}")

---
## Takeaways
* Autograd = record each operation with its inputs, then apply local derivative rules in reverse topological order.
* Gradients must accumulate (+=) because values can be used more than once.
* PyTorch's autograd does exactly this, on tensors instead of scalars.